In [1]:
import os
import shutil
import pandas as pd
import great_expectations as gx
from great_expectations.core.batch import BatchRequest
from great_expectations.data_context.types.base import DataContextConfig
from great_expectations.core.expectation_suite import ExpectationConfiguration
import json
from datetime import datetime


In [2]:
df = pd.read_csv('dataset.csv')
print(f'Строк и столбцов: {df.shape}')

Строк и столбцов: (114000, 21)


In [3]:
# удалим ненужные колонки
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(f'Строк и столбцов после очистки: {df.shape}')

Строк и столбцов после очистки: (114000, 20)


In [4]:
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())
n_genres = len(UNIQUE_GENRES)
print(f'Уникальных жанров: {n_genres}')

Уникальных жанров: 114


In [ ]:
context_dir = 'gx'
if os.path.exists(context_dir):
    shutil.rmtree(context_dir)
os.makedirs(context_dir, exist_ok=True)

context = gx.get_context(mode="file", project_root_dir=context_dir)

datasource = context.sources.add_pandas("music_data_source")

data_asset = datasource.add_dataframe_asset(name="music_data_asset")

batch_request = data_asset.build_batch_request(dataframe=df)

suite_name = "music_data_expectations"
existing_suites = [s.name for s in context.list_expectation_suites()]

# проверка - если suite_name существует
if suite_name in existing_suites:
    print(f'Suite "{suite_name}" уже существует, используем его')
    expectation_suite = context.get_expectation_suite(suite_name)
else:
    print(f'Создаём новый suite "{suite_name}"')
    expectation_suite = context.add_expectation_suite(suite_name)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="music_data_expectations"
)

Создаём новый suite "music_data_expectations"


In [6]:
# функции для добавления ожиданий
def add_type_and_length(suite, column, type_name, length):
    exp1 = ExpectationConfiguration(
        expectation_type="expect_column_values_to_be_of_type",
        kwargs={"column": column, "type_": type_name}
    )
    suite.add_expectation(exp1)
    exp2 = ExpectationConfiguration(
        expectation_type="expect_column_value_lengths_to_equal",
        kwargs={"column": column, "value": length}
    )
    suite.add_expectation(exp2)

def add_type_and_length_range(suite, column, type_name, min_len, max_len):
    exp1 = ExpectationConfiguration(
        expectation_type="expect_column_values_to_be_of_type",
        kwargs={"column": column, "type_": type_name}
    )
    suite.add_expectation(exp1)
    exp2 = ExpectationConfiguration(
        expectation_type="expect_column_value_lengths_to_be_between",
        kwargs={"column": column, "min_value": min_len, "max_value": max_len}
    )
    suite.add_expectation(exp2)

def add_type_and_range(suite, column, type_name, min_val, max_val, strict_min=False, strict_max=False):
    exp1 = ExpectationConfiguration(
        expectation_type="expect_column_values_to_be_of_type",
        kwargs={"column": column, "type_": type_name}
    )
    suite.add_expectation(exp1)
    exp2 = ExpectationConfiguration(
        expectation_type="expect_column_values_to_be_between",
        kwargs={"column": column, "min_value": min_val, "max_value": max_val, "strict_min": strict_min, "strict_max": strict_max}
    )
    suite.add_expectation(exp2)

In [7]:
# проверка обязательных колонок
required_columns = [
    'track_id', 'artists', 'album_name', 'track_name', 'popularity',
    'duration_ms', 'explicit', 'danceability', 'energy', 'key',
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'
]

for col in required_columns:
    expectation = ExpectationConfiguration(
        expectation_type="expect_column_to_exist",
        kwargs={"column": col}
    )
    expectation_suite.add_expectation(expectation)

In [8]:
# проверка на отсутствие пропусков
for col in required_columns:
    expectation = ExpectationConfiguration(
        expectation_type="expect_column_values_to_not_be_null",
        kwargs={"column": col}
    )
    expectation_suite.add_expectation(expectation)

In [9]:
add_type_and_length(expectation_suite, 'track_id', 'str', 22)

In [10]:
add_type_and_length_range(expectation_suite, 'artists', 'str', 2, 512)
add_type_and_length_range(expectation_suite, 'album_name', 'str', 2, 512)
add_type_and_length_range(expectation_suite, 'track_name', 'str', 2, 512)

In [11]:
add_type_and_range(expectation_suite, 'popularity', 'int', 0, 100)
add_type_and_range(expectation_suite, 'duration_ms', 'int', 0, 5237760, strict_min=True)

In [12]:
expectation = ExpectationConfiguration(
    expectation_type="expect_column_values_to_be_of_type",
    kwargs={"column": "explicit", "type_": "bool"}
)
expectation_suite.add_expectation(expectation);

In [13]:
add_type_and_range(expectation_suite, 'danceability', 'float', 0, 1)
add_type_and_range(expectation_suite, 'energy', 'float', 0, 1)
add_type_and_range(expectation_suite, 'key', 'int', 0, 11)
add_type_and_range(expectation_suite, 'loudness', 'float', -45, 5)
add_type_and_range(expectation_suite, 'mode', 'float', 0, 1)
add_type_and_range(expectation_suite, 'speechiness', 'float', 0, 1)
add_type_and_range(expectation_suite, 'acousticness', 'float', 0, 1)
add_type_and_range(expectation_suite, 'instrumentalness', 'float', 0, 1)
add_type_and_range(expectation_suite, 'liveness', 'float', 0, 1)
add_type_and_range(expectation_suite, 'valence', 'float', 0, 1)
add_type_and_range(expectation_suite, 'tempo', 'float', 0, 256)
add_type_and_range(expectation_suite, 'time_signature', 'int', 0, 5)

In [14]:
# track_genre проверка на уникальные жанры
exp1 = ExpectationConfiguration(
    expectation_type="expect_column_values_to_be_of_type",
    kwargs={"column": "track_genre", "type_": "str"}
)
expectation_suite.add_expectation(exp1)

exp2 = ExpectationConfiguration(
    expectation_type="expect_column_distinct_values_to_be_in_set",
    kwargs={"column": "track_genre", "value_set": list(UNIQUE_GENRES)}
)
expectation_suite.add_expectation(exp2)

exp3 = ExpectationConfiguration(
    expectation_type="expect_column_unique_value_count_to_be_between",
    kwargs={"column": "track_genre", "min_value": n_genres, "max_value": n_genres, "strict_min": False, "strict_max": False}
)
expectation_suite.add_expectation(exp3);

In [15]:
# Сохраняем suite в JSON файл
suite_json = expectation_suite.to_json_dict()
with open('music_data_expectations.json', 'w', encoding='utf-8') as f:
    json.dump(suite_json, f, indent=2, ensure_ascii=False)

In [16]:
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=expectation_suite
)

validation_result = validator.validate()

# сохраняем результат в JSON
results_dict = validation_result.to_json_dict()
with open('music_data_validation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_dict, f, indent=2, ensure_ascii=False)

Calculating Metrics:   0%|          | 0/123 [00:00<?, ?it/s]

In [17]:
# вывод результата
print('\n=== Результаты проверки ===' )
print(f'Все проверки прошли успешно: {validation_result.success}')
print(f'Всего проверок: {validation_result.statistics.get("evaluated_expectations", 0)}')
print(f'Успешных проверок: {validation_result.statistics.get("successful_expectations", 0)}')
print(f'Неуспешных проверок: {validation_result.statistics.get("unsuccessful_expectations", 0)}')


=== Результаты проверки ===
Все проверки прошли успешно: False
Всего проверок: 80
Успешных проверок: 67
Неуспешных проверок: 13
